# DSC 291: Learning Theory - PA5

**James Doan**

## Part A: Sparse Linear Predictors

### 1. What Homework 4 already gives you.

**Statement.** Discuss agnostic sample complexity assuming ERM over $\mathcal{H}_{d,k}$ is computationally available; give brute-force runtime up to poly factors and identify $k$ regimes where this is polynomial in $d$.

**Proof.** Use VC generalization bound and combinatorial counting of supports.

1. Recall VC dimension definition: $\mathrm{VCdim}(\mathcal{H}_{d,k}) \le O(k \log(e d / k))$ for $1 \le k \le d/2$, and $\mathrm{VCdim}(\mathcal{H}_{d,k}) \le O(d)$ otherwise.
2. Agnostic sample complexity ($0-1$ loss): To obtain error $\varepsilon$ with confidence $1-\delta$, ERM needs
   $$
   m = O\left(\frac{\mathrm{VCdim} + \log(1/\delta)}{\varepsilon^2}\right).
   $$
   Substituting the bound into the sparse regime gives
   $$
   m = O\left(\frac{k \log(e d/k) + \log(1/\delta)}{\varepsilon^2}\right).
   $$
   Otherwise, substituting into the dense regime gives
   $$
   m = O\left(\frac{d + \log(1/\delta)}{\varepsilon^2}\right).
   $$
3. Brute-force realizable (from HW4): Enumerate supports $T \subseteq [d]$ with $|T| \le k$ (count $\sum_{i=0}^k \binom{d}{i}$). For each $T$, solve a linear feasibility problem over weights supported on $T$; runtime up to $poly(m,d)$ times the support count.
4. Polynomial regimes: If $k$ is constant or $k = O(\log d)$, then $\binom{d}{k}$ is $poly(d)$; otherwise the enumeration is quasi/fully exponential.
5. Statistical vs computational: VC/sample complexity remains polynomial even when $k$ grows (e.g., $k = \Theta(d)$), but brute-force ERM is infeasible unless $k$ is small—hence a gap between information-theoretic learnability and computational tractability. $\blacksquare$

### 2. Hardness of agreement via set cover.

**Statement.** Reduce $\mathrm{SETCOVER}$ to $\mathrm{AGREEMENT}_{\mathcal{H}_{d,k}}$ for $k$-sparse homogeneous linear classifiers; prove correctness and conclude agnostic proper hardness (assuming $\mathrm{NP} \ne \mathrm{RP}$) via the Week 5 learner-to-agreement theorem.

**Proof.** Encode sets as coordinates; negative weights select sets; positives penalize unselected; multiplicities enforce coverage.

1. Set cover instance: Universe $U=\{u_1,\dots,u_r\}$, sets $A_1,\dots,A_q \subseteq U$, budget $k \le q$. Take $d=q$ coordinates (one per set).
2. Classifier parameter: Choose $w \in \mathbb{R}^q$ with sparsity constraint $\|w\|_0 \le k$. Interpret $w_j < 0$ as selecting set $A_j$; unselected indices have $w_j \ge 0$.
3. Sample construction (multiset allowed):
   - For each set index $j \in [q]$, add one **positive** example $(x^{(+)}_j, +1)$ where $x^{(+)}_j = e_j$ (standard basis). These charge the number of selected sets, since $\langle w, e_j \rangle = w_j$.
   - For each element $u \in U$, add $M = q+1$ copies of a **negative** example $(x^{(-)}_u, -1)$ where $(x^{(-)}_u)_j = -1$ if $u \in A_j$ and $0$ otherwise. These reward coverage: if some chosen set covers $u$, then a negative coordinate is present to offset the sign.
4. Threshold $K$: Set $K = q$ (all positives) $+ \; rM$ (all negatives).
5. Completeness (set cover $\Rightarrow$ agreement): If $\exists R \subseteq [q]$ with $|R| \le k$ covering $U$, define $w_j = +1$ for $j \in R$ and $0$ otherwise.
   - Positives: $\langle w, e_j \rangle = 1$ for $j \in R$ and $0$ otherwise, so with $\operatorname{sign}(0)=+1$ all positives are labeled $+1$.
   - Negatives: $\langle w, x^{(-)}_u \rangle = -\sum_{j \in R: u \in A_j} 1 \le -1$ by coverage, so label $-1$ is correct. Hence all $q + rM$ examples are correct; agreement holds.
6. Soundness (agreement $\Rightarrow$ set cover): Suppose some $k$-sparse $w$ labels at least $K = q + rM$ examples correctly. All $q$ positives must be correct, so $w_j \ge 0$ for all $j$ (a negative $w_j$ would misclassify $(e_j,+1)$). Sparsity implies at most $k$ indices with $w_j>0$. For each negative $(x^{(-)}_u,-1)$ to be correct, we need $\langle w, x^{(-)}_u \rangle < 0$, i.e., $\exists j$ with $w_j>0$ and $u \in A_j$. Thus the positive-support set $R = \{j : w_j>0\}$ covers $U$ with $|R| \le k$.
7. Therefore $\mathrm{SETCOVER}$ reduces to $\mathrm{AGREEMENT}_{\mathcal{H}_{d,k}}$. By the Week 5 learner-to-agreement theorem, efficient proper agnostic learning of $k$-sparse homogeneous linear classifiers (with $k$ as input) would give RP algorithms for SETCOVER; if $\mathrm{NP} \ne \mathrm{RP}$, no such learner exists. $\blacksquare$

### 3. Experiment and interpretation.

**Design.** Compare exhaustive $k$-sparse $0-1$ search (tiny $d,k$) versus an $\ell_1$-regularized convex surrogate (logistic) on synthetic data. Report agreement, surrogate loss, runtime.

**Baselines.**
- Proper sparse $0-1$ (exact regardoing the support $T$ and approximate regarding the weights $w$ due to the grid restriction): enumerate supports $|T| \le k$, grid weights in $\{-1,0,1\}$ (or small grid), predict $\operatorname{sign}(\langle w,x\rangle)$.
- Surrogate: Logistic regression with $\ell_1$ penalty (encourages sparsity; generally improper); it can be turned into a proper learner if a post-hoc step is performed e.g., keeping only the top $k$ weights by magnitude.

**Metrics.**
- Empirical $0-1$ agreement on train/validation.
- Surrogate objective (log-loss).
- Runtime scaling as $d$ grows (fix $k$) or $k$ grows (fix $d$).

**Interpretation.** Exhaustive search scales combinatorially in $k$ (support count $\binom{d}{k}$), so runtime blows up quickly. The surrogate optimizes a convex problem and scales near-linearly in $d$ for fixed $m$, but may output a dense or moderately sparse classifier; it need not be exactly $k$-sparse (hence improper). $\blacksquare$

In [1]:
import itertools
import time
import numpy as np
from sklearn.linear_model import LogisticRegression

# Synthetic data generator
def make_data(n=200, d=20, k=3, seed=0):
    rng = np.random.default_rng(seed)
    w_true = np.zeros(d)
    supp = rng.choice(d, size=k, replace=False)
    w_true[supp] = rng.choice([-1.0, 1.0], size=k)
    X = rng.normal(size=(n, d))
    y = np.sign(X @ w_true)
    y[y == 0] = 1
    # add label noise 10%
    flip = rng.random(n) < 0.1
    y[flip] *= -1
    return X, y, w_true

# Exhaustive k-sparse grid search (tiny d,k only)
def brute_sparse(X, y, k=2, grid=(-1.0, 1.0)):
    n, d = X.shape
    best_agree, best_w = -1, None
    start = time.time()
    for r in range(k + 1):
        for T in itertools.combinations(range(d), r):
            for weights in itertools.product(grid, repeat=r):
                w = np.zeros(d)
                for idx, val in zip(T, weights):
                    w[idx] = val
                preds = np.sign(X @ w)
                preds[preds == 0] = 1
                agree = np.mean(preds == y)
                if agree > best_agree:
                    best_agree, best_w = agree, w.copy()
    runtime = time.time() - start
    return best_agree, best_w, runtime

# L1-logistic surrogate
def l1_logistic(X, y, C=1.0):
    clf = LogisticRegression(
        solver='liblinear',
        l1_ratio=1,
        C=C,
        fit_intercept=False,
        max_iter=200
    )
    clf.fit(X, y)
    preds = clf.predict(X)
    agree = np.mean(preds == y)
    return agree, clf.coef_.ravel(), clf

# Experiment
X, y, w_true = make_data(n=120, d=12, k=2, seed=1)
brute_agree, brute_w, brute_time = brute_sparse(X, y, k=2, grid=(-1.0, 1.0))
log_agree, log_w, log_clf = l1_logistic(X, y, C=1.0)

print(f'{{"brute_agree": {brute_agree}, "brute_time_sec": {round(brute_time, 3)}}}')
print(f'{{"log_agree": {log_agree}, "log_sparsity": {int(np.count_nonzero(log_w))}}}')


{"brute_agree": 0.9, "brute_time_sec": 0.021}
{"log_agree": 0.9083333333333333, "log_sparsity": 8}


## Part B: Convex Fixed-Feature Learning and Its Limits

### 1. Hinge loss with an $\ell_1$ constraint as a linear program.

**Statement.** Formulate the $\ell_1$-constrained hinge-risk minimization as an LP using $w_j^+, w_j^-$ and slacks $\xi_i$.

**LP.** Variables: $w_j^+, w_j^- \ge 0$ for $j \in [d]$, $\xi_i \ge 0$ for $i \in [m]$. Set $w_j = w_j^+ - w_j^-$.

Minimize the empirical risk as a linear program $\frac{1}{m} \sum_{i=1}^m \xi_i$, convex $ \in w$, subject to:
1. Hinge constraints: $1 - y_i \sum_{j=1}^d x_{ij}(w_j^+ - w_j^-) \le \xi_i$ for all $i$.
2. Nonnegativity: $w_j^+, w_j^- \ge 0$, $\xi_i \ge 0$.
3. $\ell_1$ constraint: $\sum_{j=1}^d (w_j^+ + w_j^-) \le B$.

**Roles.** 
1. Constraint (1) linearizes $(1 - y_i \langle w, x_i \rangle)_+$
2. Constraint (2) enforces slack/variable domains
3. Constraint (3) enforces $\|w\|_1 \le B$.

**Why not $0-1$ agreement?** The LP optimizes hinge loss, a convex upper bound on $0-1$ loss; solutions need not be proper $k$-sparse (may be dense) and minimize a surrogate, not agreement. $\blacksquare$

### 2. Concrete counterexample: surrogate comparator can be bad.

**Statement.** Distribution $\mathcal{D}_{p,M}$ on $(x,y)$ with $(1,+1)$ w.p. $1-p$ and $(-M,+1)$ w.p. $p$, where $p \in (0,1/2)$ and $M > (1-p)/p$. Show hinge minimizers have $0-1$ risk $1-p$ while optimal $0-1$ risk is $p$.

**Proof.** Compute hinge risk piecewise in $w$; analyze minimizers; compare to $0-1$.

1. Hinge risk: for $f_w(x)=wx$, $$L_{\mathrm{hinge}}(w)= (1-p) (1 - w)_+ + p (1 + Mw)_+$$
2. Piecewise regions:
    $$
    L(w) =
    \begin{cases}
    p(1 + Mw), & w \ge 1, \\[6pt]
    (1-p)(1-w) + p(1+Mw), & -\frac{1}{M} \le w \le 1, \\[6pt]
    (1-p)(1-w), & w \le -\frac{1}{M}
    \end{cases}
    $$
    Which simplify to:
    $$
    L(w) =
    \begin{cases}
    p(1 + Mw), & w \ge 1, \\[6pt]
    1 + p + w(pM - (1-p)), & -\frac{1}{M} \le w \le 1, \\[6pt]
    (1-p)(1-w), & w \le -\frac{1}{M}
    \end{cases}
    $$
3. Slopes: 
   - Region (b) slope is $pM - (1-p)$. Given $M > (1-p)/p$, slope $>0$, so $L$ increases with $w$ in region (b).
   - Region (c) slope is $-(1-p)<0$, so $L$ decreases as $w$ decreases; minimum in region (c) occurs at boundary $w=-1/M$.
   - Region (b) minimum at left endpoint $w=-1/M$ as well.
   - Region (a) increases.
   - Thus any minimizer is $w^* = -1/M$ (unique or minimal set).
4. Hinge minimizer $w^*=-1/M$ yields predictions:
   - $f_{w^*}(1)= -1/M <0$ (predicts $-1$) on majority example,
   - $f_{w^*}(-M)= (+1)$ on minority example,
   - Thus $0-1$ risk = $\mathbb{P}[y \ne \operatorname{sign}(f)] = 1-p$.
5. Best $0-1$ risk: taking $w=1$ yields predictions $+1$ on both points, so error only on $(-M,+1)$ with probability $p$. Hence $\inf_w L^{0-1}_{\mathcal{D}}(f_w)=p$.
6. For any $\varepsilon>0$ and $\alpha<1$, choose $p<\varepsilon$ and $M>(1-p)/p$; then $0-1$ optimum $\le \varepsilon$ but every hinge minimizer has $0-1$ risk $1-p>\alpha$ (for suitable $p$). $\blacksquare$

### 3. Fixed-feature parity barrier: proof and tightness.

**Statement.** If a fixed feature map $\phi: \{\pm1\}^d \to \mathbb{R}^D$ represents every parity exactly by a linear predictor, then $D \ge 2^d$. Conversely, $D=2^d$ suffices via parity features.

**Proof.** Use the parity matrix rank argument.

1. Form matrix $H \in \mathbb{R}^{2^d \times 2^d}$ with rows indexed by $I \subseteq [d]$, columns by $x \in \{\pm1\}^d$, entries $H_{I,x} = \chi_I(x)$. Rows are orthogonal because for $I \ne J$, $\sum_x \chi_I(x) \chi_J(x) = 0$; norm of each row is $2^d$. Hence $\operatorname{rank}(H)=2^d$.
2. Exact representability means: $\exists w_I \in \mathbb{R}^D$ such that $\chi_I(x) = \langle w_I, \phi(x) \rangle$ for all $I,x$. Stack rows: let $W$ be $2^d \times D$ with rows $w_I^\top$ and $\Phi$ be $D \times 2^d$ with columns $\phi(x)$. Then $H = W \Phi$.
3. Rank bound: $2^d = \operatorname{rank}(H) \le \min\{\operatorname{rank}(W), \operatorname{rank}(\Phi)\} \le D$. Thus $D \ge 2^d$.
4. Tightness: take $\phi(x) = (\chi_I(x))_{I \subseteq [d]} \in \mathbb{R}^{2^d}$. Then choose $w_I$ as the standard basis vector $e_I$; we have $\langle w_I, \phi(x) \rangle = \chi_I(x)$. So $D=2^d$ suffices.

**Interpretation.** Fixed-feature convex learning can be statistically clean, but representing all parities requires exponential dimension; feature learning (nonconvex) avoids this blow-up. $\blacksquare$

### 4. Same predictors, different optimization geometry.

**Statement.** Compare fixed linear model $f_\beta(x)=\langle\beta,x\rangle$ and two-layer linear net $f_{u,v}(x)=v\langle u,x\rangle$ under squared loss.

**Proof.** Show representational equivalence, convexity vs nonconvexity, and global minima correspondence.

1. Representation: any $\beta \in \mathbb{R}^d$ factors as $\beta = v u$ (take $v=1$, $u=\beta$); conversely $v u$ yields the same linear predictor.
2. Convexity: $L_{\mathrm{lin}}(\beta) = \frac{1}{m} \sum_i (\langle\beta,x_i\rangle - y_i)^2$ is convex quadratic in $\beta$.
3. Nonconvexity: $L_{\mathrm{net}}(u,v)$ may violate Jensen. Example: take $d=1$, data $S = ((1,1),( -1,1))$. Then $f_{u,v}(x)=vux$. Check $(u,v)=(1,1)$ and $(u,v)=(-1,-1)$ both give zero loss, but midpoint $(0,0)$ yields loss $1>0$; hence nonconvex.
4. Minima correspondence: if $\beta^*$ minimizes $L_{\mathrm{lin}}$, then for any factorization $\beta^*=v u$ we have $L_{\mathrm{net}}(u,v)=L_{\mathrm{lin}}(\beta^*)$ because predictions coincide on all $x_i$. Any lower value would contradict optimality of $\beta^*$. Thus all factorizations are global minima of the nonconvex objective.

**Takeaway.** The predictor class is identical, but optimization geometry differs: convex for fixed features, nonconvex for factored parameters; learning features can destroy convexity while leaving the hypothesis class unchanged. $\blacksquare$

## Part C: AI Usage Report

1. Codex 5.1 Max was used for proof construction. Standard AI workflows were implemented, with manual human verification of completed proofs. Theorems were extracted from the problem set and implemented in `plan.md` to take advantage of chain of thought; reasoning was also increased from low to medium. The model was also encouraged to read prior exercises to increase its context on agnostic learning and surrogate loss.
2. Since most theorems were manually cross referenced with lecture, many proof constructions were accepted. The arguments from Codex 5.1 Max proceeded in a coherent sequence, with all transitions justified appropriately.
3. The experimental code generated minor warnings, which were manually resolved. These warnings were due to the use of recently deprecated modules. In addition, formatting issues were present and were manually corrected to improve readability. A few minor adjustments were made to the usage of the terms "exact" and "approximate," based on the lecture (Codex's explanation was somewhat hand-wavy). Still, we maintain that most proof constructions are correct overall.
4. This submission was verified by comparing its claims and steps to the slide deck, checking that each major result, assumption, and definition matched the slides. For points that were unclear or where terminology differed, I cross-checked with Gemini 3 to confirm precise definitions and standard usages. This check was critical for the more hand-wavy proof steps. For example, we clarified that the Empirical Risk (ER) was defined as the objective function divided by m, since the slide deck only shows how ERM can be expressed as a linear program using surrogate losses.
5. Codex was most useful in forming the initial proof constructions by reasoning about the structured steps and producing a coherent structure that was adapted and verified. Specifically, it proposed an exensive sequence of proof steps that I then expanded with explicit justifications where the model had made some assumptions about the reader's background knowledge.
6. When trying to reference each proof to a slide, the definition of ERM did not match exactly and therefore showed ambiguity. It was confirmed through Gemini 3 that Codex 5.1 Max was correct in its analysis, but we opted to write clarify further why certain terms were written as shown, assuming the reader's background may be weaker. We then verified that these terms were then correctly named and their uses were correct.